### BLEU and ROUGE scoring

Here are the commonly accepted full forms:
* BLEU:	Bilingual Evaluation Understudy
* ROUGE:	Recall-Oriented Understudy for Gisting Evaluation

A bit of historical context:

##### BLEU
Introduced in 2002 by researchers at IBM Research.
Originally designed for machine translation evaluation.
The "Bilingual" part comes from comparing machine-translated text against human translations.
##### ROUGE
Introduced in 2004, primarily for automatic summarization evaluation.
"Recall-Oriented" reflects its emphasis on how much of the reference text is captured by the generated summary.
"Gisting" refers to capturing the gist or main idea of a document.

A useful memory aid:
* **BLEU**  → Precision-oriented
       "How much of what I generated is correct?"
* **ROUGE** → Recall-oriented
       "How much of the reference did I cover?"

That's not the complete mathematical definition, but it's the intuition most practitioners remember.

Reiterating, They do NOT understand meaning, rather focus on the frequency of occurrence of the words from the context with respect to complete list of words in the answer (BLEU) or the context (ROUGE)

In [32]:
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer

def bleu_score(ref_answer:str, gen_answer:str) -> any:
    reference = [
        ref_answer.split()
    ]

    candidate = \
        gen_answer.split()

    score = sentence_bleu(
        reference,
        candidate
    )
    return float(score)

def rouge_score(ref_answer:str, gen_answer:str) -> any:

    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rougeL'],
        use_stemmer=True
    )

    scores = scorer.score(
        ref_answer,
        gen_answer
    )

    return scores

# TEST
print(f'{bleu_score(
    "the cat sat on the mat",
    "the cat sat on the mat"
):f}')
print(rouge_score(
    "the cat sat on the mat",
    "the cat sat on the mat"
))


1.000000
{'rouge1': Score(precision=1.0, recall=1.0, fmeasure=1.0), 'rougeL': Score(precision=1.0, recall=1.0, fmeasure=1.0)}


In [37]:
print(f'{bleu_score(
    "Kubernetes is an open-source container orchestration platform.",
    "Kubernetes is an orchestration platform for managing containers, managed by Open-Source."
):f}')
print(rouge_score(
    "Kubernetes is an open-source container orchestration platform.",
    "Kubernetes is an orchestration platform for managing containers, managed by Open-Source."
))

0.000000
{'rouge1': Score(precision=0.5384615384615384, recall=0.875, fmeasure=0.6666666666666667), 'rougeL': Score(precision=0.38461538461538464, recall=0.625, fmeasure=0.4761904761904762)}


In [36]:
print(f'{bleu_score(
    "A B C D X Y Z",
    "A B C D E F G H X Y"
):f}')
print(rouge_score(
    "A B C D X Y Z",
    "A B C D E F G H X Y"
))

0.312394
{'rouge1': Score(precision=0.6, recall=0.8571428571428571, fmeasure=0.7058823529411764), 'rougeL': Score(precision=0.6, recall=0.8571428571428571, fmeasure=0.7058823529411764)}


### LLM as a Judge

| Metric | Question |
|---------|----------|
| Relevance | Did the answer address the user's question? |
| Groundedness | Can the answer be traced to the retrieved context? |
| Faithfulness | Did the answer accurately represent the retrieved context? |
| Hallucination | Did the answer invent unsupported information? |

In [1]:
from openai import OpenAI

client = OpenAI(
    api_key="anything",
    base_url="http://localhost:4000"
)


In [24]:
response = client.chat.completions.create(
    model="ollama/llama3",
    max_tokens=100,
    max_completion_tokens=100,
    messages=[
        {
            "role":"user",
            "content":"What is Kubernetes?",
        }
    ]
)

print(response.choices[0].message.content)

Kubernetes (also known as K8s) is an open-source container orchestration system for automating the deployment, scaling, and management of containerized applications. It was originally designed by Google, and is now maintained by the Cloud Native Computing Foundation (CNCF).

In simple terms, Kubernetes helps you manage a cluster of computers that are running your application's containers. A container is like a lightweight virtual machine that runs a specific process or service.

Kubernetes provides many features to help you manage your


In [29]:
judge_prompt_template = """
You are an LLM evaluation judge.

Question:
%s

Context:
%s

Answer:
%s

Definitions:

Faithfulness:
How well the answer is supported by the provided context.
0 = completely unsupported
10 = fully supported

Groundedness:
Whether every factual claim in the answer can be traced to the provided context.
0 = not grounded
10 = fully grounded

Relevance:
How well the answer addresses the question.
0 = irrelevant
10 = directly answers the question

Hallucination:
Degree of fabricated or unsupported information.
0 = no hallucination
10 = severe hallucination


Always evaluate the given Q and A on the provided context only, and Return JSON only:

{
  "faithfulness": score,
  "groundedness": score,
  "relevance": score,
  "hallucination": score
}
"""

def llm_as_judge_score(question:str, context:str, answer:str):
    judge_prompt = judge_prompt_template % (question, context, answer)
    response = client.chat.completions.create(
        model="ollama/llama3",
        messages=[
            {
                "role":"user",
                "content":judge_prompt
            }
        ]
    )
    print(response.choices[0].message.content)


In [30]:
llm_as_judge_score(
    # Question:
    'What is Kubernetes?',
    # Context:
    'Kubernetes is an open-source container orchestration platform',
    # Answer:
    'Kubernetes was created by Amazon.')

{
  "faithfulness": 0, 
  "groundedness": 0, 
  "relevance": 1, 
  "hallucination": 8
}


In [34]:
llm_as_judge_score(
    'What is Kubernetes?',
    # Context:
    'Kubernetes is an open-source container orchestration platform developed by Google.',
    'According to the context, Kubernetes is a proprietary container orchestration platform developed by Google.'
)

{
"faithfulness": 0,
"groundedness": 0,
"relevance": 10,
"hallucination": 8
}

Note: The answer is not faithful to the context because it describes Kubernetes as proprietary, which is incorrect. It's an open-source platform. This results in a low faithfulness score. The answer also contains factual claims that are not supported by the provided context, so the groundedness score is 0. The relevance score is high because the answer does address the question, but it provides incorrect information, which leads to a moderate hallucination score.


In [35]:
llm_as_judge_score(
    'What is Kubernetes?',
    # Context:
    'Kubernetes is an open-source container orchestration platform.',
    'According to the context, Kubernetes is an open-source container orchestration platform developed by Google.'
)

Here is my evaluation:

{
  "faithfulness": 9,
  "groundedness": 10,
  "relevance": 10,
  "hallucination": 0
}

My reasoning:

* Faithfulness: The answer accurately reflects the context, which states that Kubernetes is an open-source container orchestration platform. The only minor point of deviation is the mention of Google as the developer, but this is a minor detail and does not detract from the overall faithfulness to the context.
* Groundedness: All factual claims in the answer can be traced back to the provided context. There are no external information or assumptions made.
* Relevance: The answer directly addresses the question of what Kubernetes is, providing a clear and concise definition that matches the context.
* Hallucination: None, as there is no fabricated or unsupported information in the answer.


In [36]:
llm_as_judge_score(
    'What is Kubernetes',
    'Kubernetes (also known as K8s) is an open-source container orchestration system for automating the deployment, scaling, and management of containerized applications. It was originally designed by Google, and is now maintained by the Cloud Native Computing Foundation (CNCF).',
    'According to the context, Kubernetes is a containerized application hosting platform by Google.'
)

{
  "faithfulness": 2,
  "groundedness": 0,
  "relevance": 4,
  "hallucination": 8
}

Note: The answer only partially supports the provided context, mentioning "platform" which is not mentioned in the original text. The factual claim that Kubernetes was originally designed by Google is supported by the context, but this information is not exhaustive (e.g., the fact that it's now maintained by CNCF is not mentioned). The relevance score is 4 because the answer does mention the name "Kubernetes", but it doesn't directly address the question. There is some degree of hallucination as the answer goes beyond what is explicitly stated in the context.


In [37]:
llm_as_judge_score(
    'Who is maintaining Kubernetes now?',
    'Kubernetes (also known as K8s) is an open-source container orchestration system for automating the deployment, scaling, and management of containerized applications. It was originally designed by Google, and is now maintained by the Cloud Native Computing Foundation (CNCF).',
    'Kubernetes is a platform created by Google.'
)

Here is my evaluation of the answer:

{
  "faithfulness": 4,
  "groundedness": 6,
  "relevance": 3,
  "hallucination": 0
}

Explanation:

Faithfulness: The answer partially supports the provided context. While it mentions Kubernetes was originally designed by Google, it does not accurately reflect its current maintenance status.

Groundedness: Every factual claim in the answer can be traced to the provided context for the statement about Google's role in designing Kubernetes. However, the answer does not mention the Cloud Native Computing Foundation (CNCF) maintaining Kubernetes, which is an important detail.

Relevance: The answer partially addresses the question by mentioning Google's role in designing Kubernetes, but it does not directly answer who maintains Kubernetes currently.

Hallucination: There is no fabricated or unsupported information in this answer.


In [ ]:
#Context: Docker was created by Solomon Hykes.